In [1]:
import pandas as pd
import time
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

In [2]:
class TrafficFileHandler(FileSystemEventHandler):
    def __init__(self, file_path):
        self.file_path = file_path
        self.last_row_count = 0

    def on_modified(self, event):
        if event.src_path.endswith("traffic_data.csv"):
            df = pd.read_csv(self.file_path)

            new_data = df.iloc[self.last_row_count:]
            self.last_row_count = len(df)

            if not new_data.empty:
                print("\n📡 New Traffic Data:")
                print(new_data)

                congested = new_data[new_data["traffic_density"] == "High"]
                if not congested.empty:
                    print("\n🚨 CONGESTION ALERT 🚨")
                    print(congested[["road_name", "average_speed_kmph"]])


In [3]:
file_path = "traffic_input/traffic_data.csv"

event_handler = TrafficFileHandler(file_path)
observer = Observer()
observer.schedule(event_handler, path="traffic_input", recursive=False)
observer.start()

print("🚦 Monitoring traffic data... Press Ctrl+C to stop.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    observer.stop()

observer.join()


🚦 Monitoring traffic data... Press Ctrl+C to stop.
